# Mobile-Money Agent Network — Transaction Cleaning & Validation

This notebook loads the raw agent and transaction datasets, preserves a raw backup (`raw_backup`), cleans transaction fields and flags anomalies, joins the agent register safely, saves `outputs/agent_transactions_clean.csv`, and documents the cleaning log and operational recommendations.

## Setup, Raw Backup & Initial Inspection

In [1]:
import os
import numpy as np
import pandas as pd
from cleaning_pipeline import clean_agent_data

agents_raw = pd.read_csv("agents.csv")
txns_raw = pd.read_csv("agent_transactions.csv")
raw_backup = txns_raw.copy(deep=True)

print("Raw agents shape:", agents_raw.shape)
print("Raw transactions shape:", txns_raw.shape)
display(agents_raw.head())
display(txns_raw.head(10))
print("\nRaw txn_type counts (including NaN):")
print(txns_raw["txn_type"].value_counts(dropna=False))
print("\nRaw missing values per column:")
print(txns_raw.isna().sum())

Raw agents shape: (20, 3)
Raw transactions shape: (612, 5)


,agent_id,county,onboarded
0,AG001,Mombasa,2024-01-01
1,AG002,Nairobi,2024-02-01
2,AG003,Kisumu,2024-03-01
3,AG004,Kiambu,2024-04-01
4,AG005,Mombasa,2024-05-01


,txn_id,agent_id,txn_type,amount,txn_date
0,TX00001,AG009,deposit,353.0,2026-03-03
1,TX00002,AG020,deposit,1238.0,2026-04-27
2,TX00003,AG008,DEPOSIT,9129.0,2026-04-29
3,TX00004,AG009,deposit,652.0,2026-04-24
4,TX00005,AG009,withdrawal,478.0,2026-04-17
5,TX00006,AG019,withdrawal,931.0,2026-04-23
6,TX00007,AG020,deposit,2583.0,2026-04-02
7,TX00008,AG019,deposit,1797.0,2026-04-12
8,TX00009,AG012,Withdrawal,4184.0,2026-04-04
9,TX00010,AG015,DEPOSIT,1987.0,2026-04-17



Raw txn_type counts (including NaN):
txn_type
deposit       142
withdrawal    131
Deposit       128
DEPOSIT       111
Withdrawal    100
Name: count, dtype: int64

Raw missing values per column:
txn_id       0
agent_id     0
txn_type     0
amount      22
txn_date     0
dtype: int64


## Step-by-Step Cleaning via `clean_agent_data(txns, agents)`

In [2]:
clean_df, cleaning_log = clean_agent_data(txns_raw, agents_raw)

print("Cleaning Log:")
for item in cleaning_log:
    print("-", item)

print("\nCleaned DataFrame shape:", clean_df.shape)
display(clean_df.head(10))

Cleaning Log:
- Removed 11 duplicate transaction rows
- Standardised txn_type to 'Deposit'/'Withdrawal' (0 unmapped/missing rows set to 'Unknown')
- Parsed txn_date to ISO format YYYY-MM-DD (0 unparseable dates)
- Flagged and imputed 20 missing/invalid amount values with median (1778)
- Flagged 53 amount outliers using IQR fences [-1537.0, 5511.0]
- Merged with agent register (8 transactions with unknown agent_id labeled 'Unknown')

Cleaned DataFrame shape: (601, 8)


,txn_id,agent_id,txn_type,amount,txn_date,amount_imputed,is_outlier,county
0,TX00001,AG009,Deposit,353.0,2026-03-03,False,False,Nakuru
1,TX00002,AG020,Deposit,1238.0,2026-04-27,False,False,Mombasa
2,TX00003,AG008,Deposit,9129.0,2026-04-29,False,True,Kisumu
3,TX00004,AG009,Deposit,652.0,2026-04-24,False,False,Nakuru
4,TX00005,AG009,Withdrawal,478.0,2026-04-17,False,False,Nakuru
5,TX00006,AG019,Withdrawal,931.0,2026-04-23,False,False,Mombasa
6,TX00007,AG020,Deposit,2583.0,2026-04-02,False,False,Mombasa
7,TX00008,AG019,Deposit,1797.0,2026-04-12,False,False,Mombasa
8,TX00009,AG012,Withdrawal,4184.0,2026-04-04,False,False,Kisumu
9,TX00010,AG015,Deposit,1987.0,2026-04-17,False,False,Nairobi


## Join Comparison (Inner vs. Left vs. Outer) & Validation

In [3]:
dedup_txns = txns_raw.drop_duplicates().copy()
dedup_txns["amount_num"] = pd.to_numeric(
    dedup_txns["amount"].astype("string").str.strip().str.replace(",", "", regex=False),
    errors="coerce",
)
dedup_txns["amount_for_loss_calc"] = dedup_txns["amount_num"].fillna(dedup_txns["amount_num"].median())

inner_join = dedup_txns.merge(agents_raw, on="agent_id", how="inner", validate="many_to_one")
left_join = dedup_txns.merge(agents_raw, on="agent_id", how="left", validate="many_to_one", indicator=True)
outer_join = dedup_txns.merge(agents_raw, on="agent_id", how="outer", validate="many_to_one")

print("Inner join rows:", len(inner_join))
print("Left join rows:", len(left_join))
print("Outer join rows:", len(outer_join))

unmatched_txns = left_join[left_join["_merge"] == "left_only"]
print("\nUnmatched transactions (e.g. AG099) count:", len(unmatched_txns))
print("Raw numeric sum of unmatched transactions:", unmatched_txns["amount_num"].sum())
print("Imputed-adjusted sum of unmatched transactions:", unmatched_txns["amount_for_loss_calc"].sum())
display(unmatched_txns[["txn_id", "agent_id", "txn_type", "amount", "txn_date", "_merge"]].head())

Inner join rows: 593
Left join rows: 601
Outer join rows: 601

Unmatched transactions (e.g. AG099) count: 8
Raw numeric sum of unmatched transactions: 29092.0
Imputed-adjusted sum of unmatched transactions: 29092.0


,txn_id,agent_id,txn_type,amount,txn_date,_merge
55,TX00056,AG099,DEPOSIT,8507.0,2026-04-01,left_only
68,TX00069,AG099,withdrawal,1877.0,2026-04-24,left_only
77,TX00078,AG099,deposit,1770.0,2026-04-06,left_only
237,TX00238,AG099,DEPOSIT,4944.0,2026-04-06,left_only
267,TX00268,AG099,DEPOSIT,3110.0,2026-04-09,left_only


## Export Cleaned Dataset (`outputs/agent_transactions_clean.csv`)

In [4]:
os.makedirs("outputs", exist_ok=True)
clean_df.to_csv("outputs/agent_transactions_clean.csv", index=False)
print("Saved outputs/agent_transactions_clean.csv with shape:", clean_df.shape)

Saved outputs/agent_transactions_clean.csv with shape: (601, 8)


## Cleaning Log & Operational Recommendations
- **Duplicate rows:** Exact duplicate transaction rows were removed prior to aggregation so transactions are not double-counted.
- **Transaction types & dates:** `txn_type` values were trimmed, lowercased, and mapped cleanly to `Deposit` and `Withdrawal` (with any unmapped entries marked `Unknown`), while `txn_date` values with mixed `/` and `-` separators were standardised to `YYYY-MM-DD`.
- **Missing/invalid amounts & outliers:** Non-numeric (`"N/A"`) and missing amounts were flagged in `amount_imputed` before median imputation, and high-value transactions were flagged in `is_outlier` using the IQR rule rather than silently deleted.
- **Unregistered agent IDs (`AG099`):** Using a left join instead of an inner join retains transactions tied to `AG099` with `county` set to `Unknown`, preventing silent revenue/volume loss and giving operations a clear queue to investigate unmapped agent codes.